In [10]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124


Looking in indexes: https://download.pytorch.org/whl/cu124
     ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
     ----------------- ---------------------- 1.8/4.1 MB 10.1 MB/s eta 0:00:01
     ----------------------------------- ---- 3.7/4.1 MB 9.5 MB/s eta 0:00:01
     ---------------------------------------- 4.1/4.1 MB 9.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
     ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
     ----------------- ---------------------- 1.8/4.1 MB 9.1 MB/s eta 0:00:01
     ----------------------------------- ---- 3.7/4.1 MB 9.1 MB/s eta 0:00:01
     ---------------------------------------- 4.1/4.1 MB 8.5 MB/s eta 0:00:00
     ---------------------------------------- 0.0/4.1 MB ? eta -:--:--
     ----------------- ---------------------- 1.8/4.1 MB 9.1 MB/s eta 0:00:01
     -----------------------------

  You can safely remove it manually.


In [3]:
pip install torch-geometric

  Using cached torch_geometric-2.6.1-py3-none-any.whl.metadata (63 kB)
Using cached torch_geometric-2.6.1-py3-none-any.whl (1.1 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
import torch_geometric
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import torch.nn as nn
from torch.optim import Optimizer
from torch_geometric.nn.kge import ComplEx


c:\Users\EzhilPriyadharshiniK\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [27]:
# Assuming your dataset is a DataFrame `df` with 'head', 'relation', 'tail' columns
df = pd.read_csv('data/books_graph_facts.csv')

In [15]:
def train(
    dataloader: DataLoader,
    model: ComplEx,
    optimizer: Optimizer,
    device: str,
) -> None:
    """
    General training loop for PyTorch models.
    """
    size = len(dataloader.dataset)
    print('###############Training Starts here#########################', size)
    
    model.train() 
    total_loss = total_examples = 0
    for batch, (head_index, rel_type, tail_index) in enumerate(dataloader):
        # Move head, relation, and tail tensors to device
        head_index, rel_type, tail_index = head_index.to(device), rel_type.to(device), tail_index.to(device)

        # Calculate the loss using ComplEx's loss function
        loss = model.loss(head_index, rel_type, tail_index)
        
        # Backpropagation and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * head_index.numel()
        total_examples += head_index.numel()
        if batch % 1000 == 0:
            loss_val = loss.item()
            current = batch * len(head_index)
            print(f"loss: {loss_val:>7f}  [{current:>5d}/{size:>5d}]")
    return total_loss

In [4]:
@torch.no_grad()
def test(
    dataloader: DataLoader,
    model: ComplEx,
    device: str
) -> None:
    """
    Test method for ComplEx model.
    Evaluates the model using Hits@k and MRR metrics.
    """
    model.eval()  # Set model to evaluation mode

    # Initialize lists to collect the full test set
    all_head_index = []
    all_rel_type = []
    all_tail_index = []

    # Loop over the DataLoader and collect all test data
    for batch, (head_index, rel_type, tail_index) in enumerate(dataloader):
        # Move head, relation, and tail tensors to device
        head_index, rel_type, tail_index = head_index.to(device), rel_type.to(device), tail_index.to(device)
        
        # Append each batch to the list
        all_head_index.append(head_index)
        all_rel_type.append(rel_type)
        all_tail_index.append(tail_index)

    # Concatenate all batches into a single tensor for each
    all_head_index = torch.cat(all_head_index, dim=0)
    all_rel_type = torch.cat(all_rel_type, dim=0)
    all_tail_index = torch.cat(all_tail_index, dim=0)

    # Call the built-in `test()` method from PyTorch Geometric's ComplEx model
    results = model.test(
        head_index=all_head_index,
        rel_type=all_rel_type,
        tail_index=all_tail_index
    )

    # Process and print results (e.g., Hits@k, MRR)
    print(f"Hits@{k}: {results['hits@k']}, MRR: {results['mrr']}")

    return results
       

In [5]:
from typing import Union
def hits_at_k(ranks: Union[torch.Tensor, np.ndarray, list], k: int) -> torch.Tensor:
    """
    Calculate Hits@k metric for ranking evaluation.
    """
    # Convert ranks to a torch tensor
    ranks_tensor = torch.tensor(ranks, dtype=torch.float32)
    
    # Calculate hits@k (how many of the ranks are <= k)
    return torch.mean((ranks_tensor <= k + 1).float())

In [6]:
def evaluate(
    dataloader: DataLoader, model: ComplEx, device: str, k: int = 10
) -> None:
    """
    Evaluation loop for ComplEx models. Calculates hits@k.
    """
    model.eval()  # Set model to evaluation mode
    ranks = []
    
    with torch.no_grad():
        for batch, (head_index, rel_type, tail_index) in enumerate(dataloader):
            # Move data to device
            head_index, rel_type, tail_index = head_index.to(device), rel_type.to(device), tail_index.to(device)

            # Forward pass to compute scores for the triplet
            scores = model(head_index, rel_type, tail_index)
            
            # Rank the true tail entity among all possible entities
            rank = torch.argsort(scores, descending=True).tolist().index(tail_index.item()) + 1
            ranks.append(rank)

    # Compute hits@k
    hits = hits_at_k(ranks, k=k)
    print(f"hits@{k}: {hits:>7f}")

In [28]:
# Create unique mappings for entities (from head and tail) and relations
entities = pd.concat([df['head'], df['tail']]).unique()  # Combine head and tail to get all unique entities
relations = df['relation'].unique()  # Get unique relations

In [29]:
# Create mappings to convert entities and relations into integer indices
entity_to_id = {entity: idx for idx, entity in enumerate(entities)}
relation_to_id = {relation: idx for idx, relation in enumerate(relations)}

In [30]:
# Apply the mappings to create new 'head_id', 'relation_id', and 'tail_id' columns
df['head_id'] = df['head'].map(entity_to_id)
df['relation_id'] = df['relation'].map(relation_to_id)
df['tail_id'] = df['tail'].map(entity_to_id)

In [31]:
# Assuming your dataset is stored in a pandas DataFrame 'df' with 'head', 'relation', 'tail' columns
# and has been mapped to integer indices

# Create the tensors for head, relation, and tail indices
head_index = torch.tensor(df['head_id'].values, dtype=torch.long)
rel_type = torch.tensor(df['relation_id'].values, dtype=torch.long)
tail_index = torch.tensor(df['tail_id'].values, dtype=torch.long)

In [32]:
# Step 2: Split the data into training and temp sets (80% train, 20% for validation and test)
train_head, temp_head, train_rel, temp_rel, train_tail, temp_tail = train_test_split(
    head_index, rel_type, tail_index, test_size=0.2, random_state=42
)

# Step 3: Split the temp_data into validation and test sets (50% validation, 50% test)
val_head, test_head, val_rel, test_rel, val_tail, test_tail = train_test_split(
    temp_head, temp_rel, temp_tail, test_size=0.5, random_state=42
)

In [33]:
#Create TensorDatasets for each split
train_dataset = TensorDataset(train_head, train_rel, train_tail)
val_dataset = TensorDataset(val_head, val_rel, val_tail)
test_dataset = TensorDataset(test_head, test_rel, test_tail)

In [34]:
# Step 5: Create DataLoaders for each split
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)
# Check the sizes of the splits
print(f"Training set size: {len(train_loader.dataset)}")
print(f"Validation set size: {len(val_loader.dataset)}")
print(f"Test set size: {len(test_loader.dataset)}")

Training set size: 480000
Validation set size: 60000
Test set size: 60000


In [35]:
# Set up the ComplEx model
num_nodes = len(pd.concat([df['head'], df['tail']]).unique())  # Total number of unique entities (nodes)
num_relations = len(df['relation'].unique())  # Total number of unique relations
hidden_channels = 100  # Embedding dimension

In [36]:
device = 'cuda'
# Initialize ComplEx model
model = ComplEx(num_nodes=num_nodes, num_relations=num_relations, hidden_channels=hidden_channels).to(device)

# Define optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [37]:
# Train the model
loss = train(train_loader, model, optimizer, device='cuda' if torch.cuda.is_available() else 'cpu')


loss: 0.693170  [    0/480000]
loss: 0.685113  [64000/480000]
loss: 0.613418  [128000/480000]
loss: 0.532357  [192000/480000]
loss: 0.551197  [256000/480000]
loss: 0.453016  [320000/480000]
loss: 0.505016  [384000/480000]
loss: 0.487317  [448000/480000]


In [17]:
epoch =1
rank, mrr, hits = test(test_loader, model, device='cuda')
print(f'Epoch: {epoch:03d}, Val Mean Rank: {rank:.2f}, '
        f'Val MRR: {mrr:.4f}, Val Hits@10: {hits:.4f}')

ValueError: too many values to unpack (expected 3)

In [45]:
for epoch in range(1, 2):
    
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')
    if epoch % 2 == 0:
        rank, mrr, hits = test(test_data, model, device='cuda')
        print(f'Epoch: {epoch:03d}, Val Mean Rank: {rank:.2f}, '
              f'Val MRR: {mrr:.4f}, Val Hits@10: {hits:.4f}')

    rank, mrr, hits_at_10 = test(test_data)
    print(f'Test Mean Rank: {rank:.2f}, Test MRR: {mrr:.4f}, '
        f'Test Hits@10: {hits_at_10:.4f}')

loss: 0.454314  [    0/600000]
loss: 0.418529  [ 6400/600000]
loss: 0.332985  [12800/600000]
loss: 0.402841  [19200/600000]
loss: 0.341701  [25600/600000]
loss: 0.358009  [32000/600000]
loss: 0.261552  [38400/600000]
loss: 0.314599  [44800/600000]
loss: 0.322540  [51200/600000]
loss: 0.409840  [57600/600000]
loss: 0.316693  [64000/600000]
loss: 0.345776  [70400/600000]
loss: 0.339647  [76800/600000]
loss: 0.320776  [83200/600000]
loss: 0.383929  [89600/600000]
loss: 0.447416  [96000/600000]
loss: 0.381586  [102400/600000]
loss: 0.359639  [108800/600000]
loss: 0.375785  [115200/600000]
loss: 0.433346  [121600/600000]
loss: 0.414941  [128000/600000]
loss: 0.292946  [134400/600000]
loss: 0.421640  [140800/600000]
loss: 0.422982  [147200/600000]
loss: 0.394695  [153600/600000]
loss: 0.332846  [160000/600000]
loss: 0.336213  [166400/600000]
loss: 0.413609  [172800/600000]
loss: 0.368616  [179200/600000]
loss: 0.377362  [185600/600000]
loss: 0.374982  [192000/600000]
loss: 0.296186  [198400/

TypeError: test() missing 2 required positional arguments: 'model' and 'device'

In [23]:
# Evaluate the model
evaluate(dataloader, model, device='cuda' if torch.cuda.is_available() else 'cpu', k=10)

RuntimeError: a Tensor with 64 elements cannot be converted to Scalar

In [35]:
from torch_geometric.datasets import FB15k_237
from torch_geometric.nn import ComplEx
import os.path as osp
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
#path = osp.join(osp.dirname(osp.realpath(__file__)), '..', 'data', 'FB15k')
path = os.path.join(os.getcwd(), 'data', 'FB15k')

train_data = FB15k_237(path, split='train')[0].to(device)
val_data = FB15k_237(path, split='val')[0].to(device)
test_data = FB15k_237(path, split='test')[0].to(device)

Processing...
Done!
